In [1]:
# !pip install --upgrade pip

# # ember github for reference 
!git clone https://github.com/FutureComputing4AI/EMBER2024.git
%pip install ./EMBER2024

%pip install pandas
%pip install altair

# # thrember dependencies
!pip uninstall -y signify
%pip install "signify==0.7.1"

# might need libomp installed:
!brew install libomp

fatal: destination path 'EMBER2024' already exists and is not an empty directory.
Processing ./EMBER2024
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for thrember: filename=thrember-0.1.0-py3-none-any.whl size=26779 sha256=218b1be03db85b2e4881696702a3395e32f6d56d7b5f60b4624a28206ad42402
  Stored in directory: /private/var/folders/s0/hh3zvcw56y7bxdk45kxvs7nr0000gn/T/pip-ephem-wheel-cache-yjhlxm9t/wheels/88/f3/1c/d3fabefde09d0e7d16619690fcd27e06f47d5701634f51fe78
Successfully built thrember
  Attempting uninstall: thrember
    Found existing installation: thrember 0.1.0
    Uninstalling thrember-0.1.0:
      Successfully uninstalled thrember-0.1.0
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Found existing installation: signify 0.7.1

In [2]:
# imports
import os
from pathlib import Path
import thrember
import pandas as pd
import numpy as np
import matplotlib.pylab as plt
import lightgbm as lgb
import polars as pl
import altair as alt
from sklearn.metrics import roc_auc_score, roc_curve
alt.renderers.enable("default")
alt.data_transformers.enable("default")

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DataTransformerRegistry.enable('default')

In [ ]:
ROOT = Path.cwd().parent
DATA_DIR = ROOT / "win32_data"
train_df = pd.read_parquet(DATA_DIR / "win32_detection_train_20pct.parquet")

print("rows:", len(train_df))
print("duplicate rows:", train_df.duplicated().sum())
print("duplicate sha256:", train_df["sha256"].duplicated().sum())


print("Shape:", train_df.shape)
print("\nColumns:")
print(train_df.columns.tolist())

display(train_df.head())


print("\nData types:")
print(train_df.dtypes)

print(train_df.iloc[2].strings)
print(train_df.iloc[2].general)



In [ ]:
label_counts = (
    train_df["label"]
    .value_counts()
    .rename_axis("label")
    .reset_index(name="count")
)

label_counts["category"] = label_counts["label"].map({
    0: "Benign",
    1: "Malware"
})

chart = alt.Chart(label_counts).mark_bar().encode(
    x=alt.X("category:N", title="File Classification"),
    y=alt.Y("count:Q", title="Number of Samples"),
    tooltip=["category", "count"]
).properties(
    title="Distribution of Malware and Benign Samples",
    width=500,
    height=350
)





chart

alt.Chart(...)

In [ ]:
#First inspect one imports value:


sample_imports = train_df.iloc[0]["imports"]

if isinstance(sample_imports, str):
    sample_imports = json.loads(sample_imports)

print(type(sample_imports))
print(sample_imports)

<class 'dict'>
{'KERNEL32.dll': ['GetDriveTypeA', 'GetModuleFileNameA', 'GetVersionExA', 'GetVersion', 'CompareStringA', 'GetTimeZoneInformation', 'IsBadCodePtr', 'IsBadReadPtr', 'SetUnhandledExceptionFilter', 'GetStringTypeW', 'GetStringTypeA', 'GetFileType', 'GetStdHandle', 'SetHandleCount', 'GetEnvironmentStringsW', 'GetEnvironmentStrings', 'FreeEnvironmentStringsW', 'FreeEnvironmentStringsA', 'UnhandledExceptionFilter', 'GetOEMCP', 'GetACP', 'GetCPInfo', 'LCMapStringW', 'LCMapStringA', 'GetCurrentProcess', 'HeapReAlloc', 'VirtualAlloc', 'VirtualFree', 'HeapCreate', 'HeapDestroy', 'GetEnvironmentVariableA', 'GetCommandLineA', 'GetStartupInfoA', 'FileTimeToLocalFileTime', 'FileTimeToSystemTime', 'FindNextFileA', 'RemoveDirectoryA', 'MoveFileA', 'RtlUnwind', 'DeleteFileA', 'SetEnvironmentVariableA', 'CreateDirectoryA', 'HeapFree', 'HeapAlloc', 'HeapCompact', 'TerminateProcess', 'ExitProcess', 'GetFileAttributesA', 'SetFileAttributesA', 'MoveFileExA', 'GetModuleHandleA', 'FormatMessage

In [ ]:
#Number of imported APIs per file

def parse_imports(value):
    if isinstance(value, dict):
        return value

    if isinstance(value, str):
        try:
            return json.loads(value)
        except (json.JSONDecodeError, TypeError):
            return {}

    return {}


def count_imported_apis(import_data):
    if isinstance(import_data, dict):
        total = 0

        for apis in import_data.values():
            if isinstance(apis, list):
                total += len(apis)

        return total

    if isinstance(import_data, list):
        return len(import_data)

    return 0


parsed_imports = train_df["imports"].apply(parse_imports)

api_count_df = pd.DataFrame({
    "api_count": parsed_imports.apply(count_imported_apis),
    "label": train_df["label"]
})

api_count_df["category"] = api_count_df["label"].map({
    0: "Benign",
    1: "Malware"
})

display(
    api_count_df.groupby("category")["api_count"]
    .agg(["count", "mean", "median", "std", "max"])
    .round(2)
)

,count,mean,median,std,max
category,,,,,
Benign,155768,202.50,117.0,348.65,8192
Malware,156357,106.83,84.0,127.61,4616


In [ ]:
#Because the dataset is large, aggregate the histogram before sending it to Altair:

upper_limit = api_count_df["api_count"].quantile(0.99)

api_plot_df = api_count_df[
    api_count_df["api_count"] <= upper_limit
].copy()

bin_edges = np.linspace(
    api_plot_df["api_count"].min(),
    api_plot_df["api_count"].max(),
    41
)

api_plot_df["api_bin"] = pd.cut(
    api_plot_df["api_count"],
    bins=bin_edges,
    include_lowest=True
)

api_histogram_df = (
    api_plot_df
    .groupby(["category", "api_bin"], observed=True)
    .size()
    .reset_index(name="count")
)

api_histogram_df["bin_midpoint"] = (
    api_histogram_df["api_bin"]
    .apply(lambda interval: interval.mid)
    .astype(float)
)

api_histogram_df["percentage"] = (
    api_histogram_df.groupby("category")["count"]
    .transform(lambda values: values / values.sum() * 100)
)

api_histogram_plot = api_histogram_df[
    ["category", "bin_midpoint", "count", "percentage"]
].copy()

In [ ]:
#Plot it:
alt.Chart(api_histogram_plot).mark_line(
    point=True
).encode(
    x=alt.X(
        "bin_midpoint:Q",
        title="Number of imported APIs"
    ),
    y=alt.Y(
        "percentage:Q",
        title="Percentage of files"
    ),
    color=alt.Color(
        "category:N",
        title="File type"
    ),
    tooltip=[
        "category:N",
        alt.Tooltip(
            "bin_midpoint:Q",
            title="Imported APIs",
            format=".0f"
        ),
        alt.Tooltip(
            "percentage:Q",
            title="Percentage",
            format=".2f"
        )
    ]
).properties(
    title="Imported API Count: Malware vs. Benign",
    width=650,
    height=400
)

alt.Chart(...)

In [ ]:
"""Flatten the imported API names

This converts each file’s nested imports into one list:"""
def flatten_imported_apis(import_data):
    api_names = []

    if isinstance(import_data, dict):
        for apis in import_data.values():
            if isinstance(apis, list):
                api_names.extend(
                    str(api).lower()
                    for api in apis
                )

    elif isinstance(import_data, list):
        api_names.extend(
            str(api).lower()
            for api in import_data
        )

    return api_names


api_lists = parsed_imports.apply(flatten_imported_apis)

print(api_lists.iloc[0][:20])

['getdrivetypea', 'getmodulefilenamea', 'getversionexa', 'getversion', 'comparestringa', 'gettimezoneinformation', 'isbadcodeptr', 'isbadreadptr', 'setunhandledexceptionfilter', 'getstringtypew', 'getstringtypea', 'getfiletype', 'getstdhandle', 'sethandlecount', 'getenvironmentstringsw', 'getenvironmentstrings', 'freeenvironmentstringsw', 'freeenvironmentstringsa', 'unhandledexceptionfilter', 'getoemcp']


In [ ]:
"""Compare networking API usage

Define a reasonable set of networking-related terms:"""

network_terms = [
    "socket",
    "connect",
    "send",
    "recv",
    "bind",
    "listen",
    "accept",
    "internetopen",
    "internetconnect",
    "internetreadfile",
    "internetwritefile",
    "httpopenrequest",
    "httpsendrequest",
    "urlopen",
    "urldownloadtofile",
    "winhttpopen",
    "winhttpconnect",
    "winhttpsendrequest",
    "wsastartup"
]

In [ ]:
#Count networking APIs per file:
def count_matching_apis(api_names, search_terms):
    return sum(
        any(term in api_name for term in search_terms)
        for api_name in api_names
    )


network_df = pd.DataFrame({
    "network_api_count": api_lists.apply(
        lambda names: count_matching_apis(names, network_terms)
    ),
    "label": train_df["label"]
})

network_df["category"] = network_df["label"].map({
    0: "Benign",
    1: "Malware"
})

In [ ]:
#Calculate how many files import at least one networking API:
network_summary = (
    network_df.assign(
        uses_network_api=network_df["network_api_count"] > 0
    )
    .groupby("category")
    .agg(
        files=("uses_network_api", "size"),
        files_using_networking=("uses_network_api", "sum"),
        average_network_apis=("network_api_count", "mean"),
        median_network_apis=("network_api_count", "median")
    )
    .reset_index()
)

network_summary["percentage_using_networking"] = (
    network_summary["files_using_networking"]
    / network_summary["files"]
    * 100
)

display(network_summary.round(2))

,category,files,files_using_networking,average_network_apis,median_network_apis,percentage_using_networking
0,Benign,155768,63518,2.67,0.0,40.78
1,Malware,156357,72199,2.29,0.0,46.18


In [ ]:
#Visualize the percentage:
alt.Chart(network_summary).mark_bar().encode(
    x=alt.X(
        "category:N",
        title="File type"
    ),
    y=alt.Y(
        "percentage_using_networking:Q",
        title="Files importing networking APIs (%)"
    ),
    tooltip=[
        "category:N",
        alt.Tooltip(
            "percentage_using_networking:Q",
            title="Percentage",
            format=".2f"
        ),
        "files_using_networking:Q",
        "files:Q"
    ]
).properties(
    title="Networking API Usage: Malware vs. Benign",
    width=500,
    height=350
)

alt.Chart(...)

In [ ]:
"""Compare several API categories"""
api_categories = {
    "Networking": [
        "socket", "connect", "send", "recv",
        "internetopen", "httpsendrequest",
        "winhttp", "wsastartup"
    ],

    "File operations": [
        "createfile", "readfile", "writefile",
        "deletefile", "copyfile", "movefile"
    ],

    "Registry": [
        "regopenkey", "regsetvalue", "regqueryvalue",
        "regcreatekey", "regdeletekey"
    ],

    "Process and memory": [
        "createprocess", "openprocess",
        "virtualalloc", "virtualprotect",
        "writeprocessmemory", "createremotethread"
    ],

    "Cryptography": [
        "cryptencrypt", "cryptdecrypt",
        "cryptacquirecontext", "bcrypt",
        "certopenstore"
    ]
}

In [ ]:
#Create category-level data:
category_rows = []

for category_name, terms in api_categories.items():
    counts = api_lists.apply(
        lambda names: count_matching_apis(names, terms)
    )

    for file_type, label_value in [
        ("Benign", 0),
        ("Malware", 1)
    ]:
        class_counts = counts[train_df["label"] == label_value]

        category_rows.append({
            "api_category": category_name,
            "file_type": file_type,
            "percentage_of_files": (
                (class_counts > 0).mean() * 100
            ),
            "average_count": class_counts.mean()
        })

api_category_df = pd.DataFrame(category_rows)

display(api_category_df.round(2))

,api_category,file_type,percentage_of_files,average_count
0,Networking,Benign,39.06,2.28
1,Networking,Malware,45.42,1.91
2,File operations,Benign,50.59,2.36
3,File operations,Malware,60.65,2.76
4,Registry,Benign,35.25,1.54
5,Registry,Malware,43.44,1.76
6,Process and memory,Benign,45.91,1.04
7,Process and memory,Malware,66.09,1.19
8,Cryptography,Benign,7.89,0.21
9,Cryptography,Malware,6.36,0.09


In [ ]:
# Visualize:
alt.Chart(api_category_df).mark_bar().encode(
    x=alt.X(
        "api_category:N",
        title="API category"
    ),
    xOffset="file_type:N",
    y=alt.Y(
        "percentage_of_files:Q",
        title="Files importing category (%)"
    ),
    color=alt.Color(
        "file_type:N",
        title="File type"
    ),
    tooltip=[
        "api_category:N",
        "file_type:N",
        alt.Tooltip(
            "percentage_of_files:Q",
            title="Percentage",
            format=".2f"
        ),
        alt.Tooltip(
            "average_count:Q",
            title="Average API count",
            format=".2f"
        )
    ]
).properties(
    title="Imported API Categories: Malware vs. Benign",
    width=700,
    height=400
)

alt.Chart(...)

In [ ]:
# ENTROPY LEVEL VISUALIZATION

def parse_json(value):
    if isinstance(value, (dict, list)):
        return value

    if isinstance(value, str):
        try:
            return json.loads(value)
        except (json.JSONDecodeError, TypeError):
            return {}

    return {}

In [ ]:
def find_entropy_fields(value, path=""):
    results = []

    if isinstance(value, dict):
        for key, child_value in value.items():
            current_path = f"{path}.{key}" if path else key

            if "entropy" in key.lower():
                results.append((current_path, child_value))

            results.extend(
                find_entropy_fields(child_value, current_path)
            )

    elif isinstance(value, list):
        for index, child_value in enumerate(value):
            current_path = f"{path}[{index}]"
            results.extend(
                find_entropy_fields(child_value, current_path)
            )

    return results

In [ ]:
json_columns = ["general", "strings", "imports"]

for column in json_columns:
    print(f"\n===== {column.upper()} =====")

    for row_number in range(min(5, len(train_df))):
        parsed_value = parse_json(train_df.iloc[row_number][column])
        entropy_fields = find_entropy_fields(parsed_value)

        if entropy_fields:
            print(f"\nRow {row_number}:")
            for field_path, value in entropy_fields:
                print(field_path, "=", value)


===== GENERAL =====

Row 0:
entropy = 7.98444658339403

Row 1:
entropy = 3.710459087578284

Row 2:
entropy = 7.2352091959923825

Row 3:
entropy = 6.492246788417963

Row 4:
entropy = 6.436412852928497

===== STRINGS =====

Row 0:
entropy = 6.573109920983796

Row 1:
entropy = 4.989280700683594

Row 2:
entropy = 4.89426326751709

Row 3:
entropy = 5.685435771942139

Row 4:
entropy = 4.000203636979068

===== IMPORTS =====


In [ ]:
strings_parsed = train_df["strings"].apply(parse_json)

print(strings_parsed.iloc[0])

{'numstrings': 22744, 'avlength': 5.843782975729863, 'printabledist': [1485, 1269, 1546, 1349, 1735, 1322, 1423, 1220, 1524, 1460, 1540, 1267, 1293, 1331, 1234, 1118, 1184, 1333, 1320, 1282, 1409, 1295, 1310, 1243, 1177, 1318, 1143, 1249, 1252, 1140, 1190, 1196, 1133, 1607, 1350, 1403, 1843, 1640, 1522, 1223, 1451, 1628, 1416, 1280, 1397, 1365, 1272, 1235, 1538, 1679, 1634, 1491, 1580, 1618, 1392, 1389, 1275, 1414, 1417, 1303, 1282, 1281, 1243, 1236, 1104, 1514, 1440, 1579, 1459, 2036, 1398, 1341, 1380, 1694, 1394, 1446, 1643, 1503, 1590, 1514, 1327, 1328, 1672, 1599, 1748, 1404, 1246, 1419, 1120, 1294, 1173, 1197, 1163, 1222, 1104, 1166], 'printables': 132911, 'entropy': 6.573109920983796, 'string_counts': {'.click(': 2, 'clipboard': 4, 'command': 2, 'create': 15, 'delete': 5, 'desktop': 1, 'directory': 6, 'disk': 2, 'dos_msg': 1, 'enum': 1, 'environment': 7, 'exit': 4, 'file': 35, 'http://': 2, 'install': 6, 'internet': 5, 'ipv4_addr': 2, 'memory': 1, 'module': 2, 'privilege': 3, 'pr

In [ ]:
string_entropy_df = pd.DataFrame({
    "entropy": strings_parsed.apply(
        lambda value: value.get("entropy")
        if isinstance(value, dict)
        else None
    ),
    "label": train_df["label"]
})

string_entropy_df["category"] = string_entropy_df["label"].map({
    0: "Benign",
    1: "Malware"
})

string_entropy_df["entropy"] = pd.to_numeric(
    string_entropy_df["entropy"],
    errors="coerce"
)

string_entropy_df = string_entropy_df.dropna(
    subset=["entropy", "category"]
)

print("Rows available:", len(string_entropy_df))
display(string_entropy_df.head())

Rows available: 312125


,entropy,label,category
0,6.573110,0,Benign
1,4.989281,0,Benign
2,4.894263,1,Malware
3,5.685436,0,Benign
4,4.000204,1,Malware


In [ ]:
string_entropy_summary = (
    string_entropy_df
    .groupby("category")["entropy"]
    .agg([
        "count",
        "mean",
        "median",
        "std",
        "min",
        "max"
    ])
    .round(4)
)

display(string_entropy_summary)

,count,mean,median,std,min,max
category,,,,,,
Benign,155768,5.7192,5.7346,0.5488,0.0,6.5849
Malware,156357,5.7926,5.9112,0.7225,0.0,6.5850


In [ ]:
percentiles = (
    string_entropy_df
    .groupby("category")["entropy"]
    .quantile([0.01, 0.25, 0.50, 0.75, 0.99])
    .unstack()
    .round(4)
)

percentiles.columns = [
    "1%",
    "25%",
    "50%",
    "75%",
    "99%"
]

display(percentiles)

,1%,25%,50%,75%,99%
category,,,,,
Benign,3.6272,5.5025,5.7346,6.0034,6.5848
Malware,3.3963,5.3768,5.9112,6.4309,6.5844


In [ ]:
def create_entropy_histogram_data(
    data,
    value_column="entropy",
    bins=50
):
    plot_data = data[
        ["category", value_column]
    ].dropna().copy()

    minimum = plot_data[value_column].min()
    maximum = plot_data[value_column].max()

    bin_edges = np.linspace(
        minimum,
        maximum,
        bins + 1
    )

    plot_data["entropy_bin"] = pd.cut(
        plot_data[value_column],
        bins=bin_edges,
        include_lowest=True
    )

    histogram_data = (
        plot_data
        .groupby(
            ["category", "entropy_bin"],
            observed=True
        )
        .size()
        .reset_index(name="count")
    )

    histogram_data["bin_midpoint"] = (
        histogram_data["entropy_bin"]
        .apply(lambda interval: interval.mid)
        .astype(float)
    )

    histogram_data["percentage"] = (
        histogram_data
        .groupby("category")["count"]
        .transform(
            lambda values: values / values.sum() * 100
        )
    )

    return histogram_data[
        [
            "category",
            "bin_midpoint",
            "count",
            "percentage"
        ]
    ]


entropy_histogram_df = create_entropy_histogram_data(
    string_entropy_df,
    bins=50
)

In [ ]:


entropy_histogram = (
    alt.Chart(entropy_histogram_df)
    .mark_line(point=True)
    .encode(
        x=alt.X(
            "bin_midpoint:Q",
            title="String entropy"
        ),
        y=alt.Y(
            "percentage:Q",
            title="Percentage of files"
        ),
        color=alt.Color(
            "category:N",
            title="File type"
        ),
        tooltip=[
            "category:N",
            alt.Tooltip(
                "bin_midpoint:Q",
                title="Entropy",
                format=".3f"
            ),
            alt.Tooltip(
                "percentage:Q",
                title="Percentage",
                format=".2f"
            )
        ]
    )
    .properties(
        title="String Entropy Distribution: Malware vs. Benign",
        width=650,
        height=400
    )
)

entropy_histogram

alt.Chart(...)

In [ ]:
print(train_df.columns.tolist())

['sha256', 'label', 'general', 'strings', 'imports']


In [ ]:


def parse_json(value):
    if isinstance(value, dict):
        return value

    if isinstance(value, str):
        try:
            return json.loads(value)
        except (json.JSONDecodeError, TypeError):
            return {}

    return {}


# Extract general entropy
general_parsed = train_df["general"].apply(parse_json)

general_entropy_df = pd.DataFrame({
    "entropy": general_parsed.apply(
        lambda row: row.get("entropy")
        if isinstance(row, dict)
        else None
    ),
    "label": train_df["label"]
})

general_entropy_df["category"] = general_entropy_df["label"].map({
    0: "Benign",
    1: "Malware"
})

general_entropy_df["entropy"] = pd.to_numeric(
    general_entropy_df["entropy"],
    errors="coerce"
)

general_entropy_df = general_entropy_df.dropna(
    subset=["entropy", "category"]
)

display(
    general_entropy_df
    .groupby("category")["entropy"]
    .agg(["count", "mean", "median", "std", "min", "max"])
    .round(4)
)

,count,mean,median,std,min,max
category,,,,,,
Benign,155768,6.2947,6.4856,1.1304,0.0003,8.0
Malware,156357,6.6423,7.0050,1.3876,0.0007,8.0


In [ ]:
# Create entropy bins
bin_edges = np.linspace(
    general_entropy_df["entropy"].min(),
    general_entropy_df["entropy"].max(),
    51
)

general_entropy_df["entropy_bin"] = pd.cut(
    general_entropy_df["entropy"],
    bins=bin_edges,
    include_lowest=True
)

# Count samples in each bin
general_histogram_df = (
    general_entropy_df
    .groupby(
        ["category", "entropy_bin"],
        observed=True
    )
    .size()
    .reset_index(name="count")
)

# Convert each interval into a numerical midpoint
general_histogram_df["bin_midpoint"] = (
    general_histogram_df["entropy_bin"]
    .apply(lambda interval: interval.mid)
    .astype(float)
)

# Normalize each class to percentages
general_histogram_df["percentage"] = (
    general_histogram_df
    .groupby("category")["count"]
    .transform(lambda values: values / values.sum() * 100)
)

# Remove Interval objects before passing data to Altair
general_histogram_plot = general_histogram_df[
    ["category", "bin_midpoint", "count", "percentage"]
].copy()

In [ ]:
general_entropy_chart = (
    alt.Chart(general_histogram_plot)
    .mark_bar(opacity=0.55)
    .encode(
        x=alt.X(
            "bin_midpoint:Q",
            title="General entropy",
            bin=alt.Bin(step=0.15)
        ),
        y=alt.Y(
            "percentage:Q",
            title="Percentage of files",
            stack=None
        ),
        color=alt.Color(
            "category:N",
            title="File type"
        ),
        tooltip=[
            "category:N",
            alt.Tooltip(
                "bin_midpoint:Q",
                title="Entropy",
                format=".3f"
            ),
            alt.Tooltip(
                "percentage:Q",
                title="Percentage",
                format=".2f"
            ),
            alt.Tooltip(
                "count:Q",
                title="Number of files"
            )
        ]
    )
    .properties(
        title="General Entropy Distribution: Malware vs. Benign",
        width=650,
        height=400
    )
)

general_entropy_chart

alt.Chart(...)